# synthetic-gov-data-kit — Quickstart

Generate structured synthetic US government benefits test cases for LLM reasoning evaluation.

**Part of the [CivBench](https://github.com/civbench) ecosystem.**

---

## What you'll do in this notebook

1. Inspect the verified FY2026 SNAP income/benefit limits
2. Generate SNAP eligibility test cases for Virginia (edge-saturated)
3. Inspect a case — scenario, task, expected answer, and rationale trace
4. Export to CivBench YAML, JSONL, and CSV
5. Run a batch across multiple states
6. Score a mock model output with the RationaleEvaluator

In [ ]:
# Install if needed
# !pip install synthetic-gov-data-kit

In [ ]:
import sys, os
# If running from the repo root:
sys.path.insert(0, os.path.abspath('..'))

from govsynth import Pipeline, BatchPipeline, list_presets
from govsynth.sources.us.snap import SNAPSource
from govsynth.evaluation import RationaleEvaluator

print('govsynth imported successfully')

## 1. Available Presets

In [ ]:
list_presets()

## 2. Inspect Verified FY2026 SNAP Limits

In [ ]:
source = SNAPSource(fiscal_year=2026, state='VA')
t = source.thresholds()

print(f'Program: {t.program.upper()}')
print(f'Period: {source.fy_config.period_label} ({source.fy_config.citation_prefix()})')
print(f'FPL basis: {source.fy_config.fpl_year} HHS poverty guidelines')
print(f'Verification: {t.extra["verification_status"].upper()}')
print(f'Source: {t.source}')
print()
print(f'{"HH Size":<10} {"Gross (130% FPL)":<22} {"Net (100% FPL)":<20} {"Max Benefit"}')
print('-' * 65)
for size in range(1, 9):
    lim = t.by_household_size(size)
    print(f'{size:<10} ${lim.gross_monthly:>8,.0f}/month        ${lim.net_monthly:>8,.0f}/month    ${lim.max_benefit:>6,.0f}')
print()
print(f'Asset limits: ${t.asset_limit_general:,} general / ${t.asset_limit_elderly_disabled:,} elderly+disabled')
print(f'BBCE state (no asset test): {t.extra["bbce_state"]}')

## 3. Generate 20 SNAP Cases (Virginia, Edge-Saturated)

In [ ]:
pipeline = Pipeline.from_preset('snap.va')
cases = pipeline.generate(n=20, seed=42)

print(f'\nGenerated {len(cases)} cases')
print(f'Eligible:   {sum(1 for c in cases if c.expected_outcome == "eligible")}')
print(f'Ineligible: {sum(1 for c in cases if c.expected_outcome == "ineligible")}')

# Difficulty breakdown
from collections import Counter
diff_counts = Counter(c.difficulty.value for c in cases)
print(f'Difficulty: {dict(diff_counts)}')

## 4. Inspect a Single Case

In [ ]:
# Find a 'hard' eligible case to inspect
hard_cases = [c for c in cases if c.difficulty.value == 'hard']
case = hard_cases[0] if hard_cases else cases[0]

print('=== CIVBENCH ID ===')
print(case.civbench_id)
print()
print('=== SCENARIO ===')
print(case.scenario.summary)
print(f'  HH size: {case.scenario.household_size}, Gross: ${case.scenario.monthly_gross_income:,.2f}, '
      f'Net: ${case.scenario.monthly_net_income:,.2f}, Assets: ${case.scenario.liquid_assets:,.2f}')
print()
print('=== TASK ===')
print(case.task.instruction[:200] + '...')
print()
print('=== EXPECTED OUTCOME ===')
print(case.expected_outcome.upper())
print()
print('=== EXPECTED ANSWER ===')
print(case.expected_answer)
print()
print('=== RATIONALE TRACE ===')
print(case.rationale_trace.to_plain_text())
print()
print('=== VARIATION TAGS ===')
print(case.variation_tags)

## 5. Export to Multiple Formats

In [ ]:
import os
os.makedirs('./output', exist_ok=True)

# CivBench YAML (one file per case)
pipeline.save(cases, './output/snap_va/', formats=['civbench_yaml'])

# JSONL for fine-tuning
pipeline.save(cases, './output/snap_va.jsonl', formats=['jsonl'])

# CSV for review
pipeline.save(cases, './output/snap_va.csv', formats=['csv'])

print('\nOutput files:')
for f in os.listdir('./output'):
    print(f'  {f}')

## 6. Generate Across Multiple States (Batch)

In [ ]:
batch = BatchPipeline.from_presets(['snap.va', 'snap.ca', 'snap.tx'])
all_cases = batch.generate(n_per_pipeline=15, seed=42)

# Show jurisdiction breakdown
from collections import Counter
by_state = Counter(c.jurisdiction for c in all_cases)
print('Cases by jurisdiction:')
for j, n in sorted(by_state.items()):
    print(f'  {j}: {n}')

# Compare eligibility rates — CA (BBCE) should differ from TX (strict asset test)
print()
for state in ['us.va', 'us.ca', 'us.tx']:
    state_cases = [c for c in all_cases if c.jurisdiction == state]
    if state_cases:
        elig_rate = sum(1 for c in state_cases if c.expected_outcome == 'eligible') / len(state_cases)
        bbce = any('bbce_state' in c.variation_tags for c in state_cases)
        print(f'  {state}: {elig_rate:.0%} eligible, BBCE={bbce}')

## 7. Score a Mock Model Output with RationaleEvaluator

In [ ]:
evaluator = RationaleEvaluator()

# Use the hard case from section 4
test_case = hard_cases[0] if hard_cases else cases[0]

# Mock a 'good' model output that covers the key steps
good_output = f"""
Let me work through the SNAP eligibility determination step by step.

First, I need to check the gross income test under 7 CFR 273.9(a)(1).
The household has {test_case.scenario.household_size} members with a gross income of 
${test_case.scenario.monthly_gross_income:,.2f}/month.
The 130% FPL limit for this household size is ${test_case.source.thresholds().by_household_size(min(test_case.scenario.household_size,8)).gross_monthly:,.2f}.

After applying the 20% earned income deduction and the standard deduction per 7 CFR 273.9(c),
the net income comes to ${test_case.scenario.monthly_net_income or 0:,.2f}/month.

This household is eligible for SNAP benefits.
""".strip() if test_case.source else "The household is eligible."

# Actually just use a simple mock
good_output = (
    f"The household has {test_case.scenario.household_size} members. "
    f"Gross income ${test_case.scenario.monthly_gross_income:,.2f} must be checked against "
    f"the 130% FPL limit per 7 CFR 273.9(a)(1). After the 20% earned income deduction "
    f"and standard deduction under 7 CFR 273.9(c)(2), net income is "
    f"${test_case.scenario.monthly_net_income or 0:,.2f}. "
    f"The asset limit of $3,000 applies per 7 CFR 273.8(b). "
    f"This household is {'eligible' if test_case.expected_outcome == 'eligible' else 'ineligible'} for SNAP."
)

score = evaluator.score(test_case, good_output)
print(score)
print(f'\nSteps covered: {score.steps_covered}')
print(f'Steps missed:  {score.steps_missed}')
print(f'Rules cited:   {score.rules_cited}')

## 8. At-Threshold Profile Factory

Generate profiles precisely at policy boundaries — the edge cases where models fail.

In [ ]:
from govsynth.profiles import USHouseholdProfile

# Generate a household exactly at the 3-person SNAP gross income limit
at_limit = USHouseholdProfile.at_threshold(
    program='snap', threshold='gross_income_limit',
    state='VA', household_size=3, fiscal_year=2026,
    offset_pct=0.0,  # exactly at limit
    seed=42,
)
print(f'AT LIMIT:    gross=${at_limit.monthly_gross_income:,.2f} (should be $2,888.00)')

# Just above the limit — should be ineligible
above = USHouseholdProfile.at_threshold(
    program='snap', threshold='gross_income_limit',
    state='VA', household_size=3, fiscal_year=2026,
    offset_pct=0.01,  # 1% above
    seed=42,
)
print(f'ABOVE LIMIT: gross=${above.monthly_gross_income:,.2f} (should be ~$2,916.88)')

# Just below the limit — should be eligible
below = USHouseholdProfile.at_threshold(
    program='snap', threshold='gross_income_limit',
    state='VA', household_size=3, fiscal_year=2026,
    offset_pct=-0.01,  # 1% below
    seed=42,
)
print(f'BELOW LIMIT: gross=${below.monthly_gross_income:,.2f} (should be ~$2,859.12)')

# Verify eligibility for each
source = SNAPSource(fiscal_year=2026, state='VA')
for label, profile in [('AT LIMIT', at_limit), ('ABOVE', above), ('BELOW', below)]:
    eligible, reason = source.is_eligible(
        household_size=profile.household_size,
        gross_income=profile.monthly_gross_income,
        net_income=profile.monthly_net_income,
        liquid_assets=profile.liquid_assets,
    )
    print(f'{label:12} → {"ELIGIBLE" if eligible else "INELIGIBLE"}: {reason[:80]}')

## Next Steps

- See `notebooks/02_snap_edge_cases.ipynb` for deeper SNAP edge case analysis
- See `notebooks/03_wic_eligibility.ipynb` for WIC case generation
- See `notebooks/04_batch_civbench_suite.ipynb` to generate a full CivBench test suite
- Run `python scripts/verify_thresholds.py` to check threshold verification status
- See [CLAUDE.md](../CLAUDE.md) for AI coding assistant context
- See [AGENTS.md](../AGENTS.md) for agentic workflow playbooks